In [ ]:
# Configuration

# Provide your Checkmarx One tenant name:
tenant_name = "my_tenant"

# Provide either the region name for your multi-tenant instance or your single-tenant server name (do not include "https"):
# For multi-tenant regional moniker shortcut names (US, US2, DEU, etc.), see: 
# https://checkmarx.stoplight.io/docs/checkmarx-one-api-reference-guide/branches/main/fm1ma9xg73dx9-authentication-api#url
server = "US" # using the US multi-tenant region

# If using an OAuth client, provide the client name and secret as strings.  If using an API Key, these can be set to None:
oauth_client = None
oauth_secret = None

# If using an API Key, provide the API key as a string.  If using an OAuth client, this can be set to None:
api_key = None

# Provide the name for this agent that will show as the source of operations such as scanning:
agent_name = "SarifTutorial"

# Provide a list of one or more scans ids for which a SARIF log will be generated.
scan_ids = []

# Set to true to display the SARIF log in the notebook cell outputs.  Displaying the SARIF log JSON may make the cell take a long time to render.
display_sarif = False


## Optional Configuration

The next cell contains optional configuration options.  Usually these are not needed.

### Viewing the Tutorial's API I/O

While running the tutorial, it may be helpful to intercept the Checkmarx One API I/O to gain a deeper understanding of how `cxone-sarif` is operating.  This can be done with `mitmproxy`.

One method of running `mitmproxy` is to run it as a docker container.  The following command line is an example of how it might be executed:

`docker run --rm -it -p 8080:8080 mitmproxy/mitmproxy mitmproxy --ssl-insecure`

Then set the `proxies` and `ssl_verify` properties in the following cell:

```python
proxies = {
  "http" : "http://localhost:8080",
  "https" : "http://localhost:8080"
}

ssl_verify=False

```

The `proxies` setting may need to be varied if running this notebook using the Jupyter Notebook server rather than VS Code.


In [ ]:
# Notebook options - mostly these can be left as-is

# Set to True to force install of the latest cxone-sarif module from the GitHub release instead of the local
# package (if running this notebook in VSCode from a clone of the cxone-sarif repository)/
force_download = False

# Proxy settings - leave at None if not using a proxy, otherwise this is a dictionary that follows the Requests API proxy settings
proxies = None

# SSL check settings - leave at True for most uses
ssl_verify=True


In [ ]:
%%capture ignore

# Determine and install the latest cxone-sarif version or the local code, don't modify this cell.

import os
from pathlib import Path

local_module_path = Path("..") / Path("pyproject.toml")

%pip uninstall -y cxone_sarif

if os.path.exists(local_module_path) and not force_download:
  %pip install ..

  from cxone_sarif.__version__ import __version__
  api_version = __version__
else:
  %pip install requests

  import requests
  latest_redirect = requests.get("https://github.com/checkmarx-ts/cxone-sarif/releases/latest", allow_redirects=False)
  assert(latest_redirect.status_code == 302)
  api_version = latest_redirect.headers['location'].split("/")[-1:].pop()
  %pip uninstall requests -y

  %pip install https://github.com/checkmarx-ts/cxone-sarif/releases/download/$api_version/cxone_sarif-$api_version-py3-none-any.whl


%pip install tqdm
from requests.packages import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)



In [ ]:
%%capture ignore
# Add some utility functions, don't modify this cell
from IPython.display import Markdown, JSON as int_JSON, display
import asyncio, json
from cxone_sarif import SerializableSarifLog

def MD(md_string):
    display(Markdown(md_string))

def pretty_print(log):
    json_str = ""
    if isinstance(log, str):
        json_str = log
    elif isinstance(log, SerializableSarifLog):
        json_str = log.asjson()
    else:
        return ""
    
    return json.dumps(json.loads(json_str), indent=2)

def JSON(log):
    if display_sarif:
        MD(f"```json\n{pretty_print(log)}\n```")

def write_log(filename, log):
    with open(filename, "wt") as sarif_out:
        sarif_out.write(pretty_print(log))



In [ ]:
MD(f"**Using cxone-sarif version {api_version}.**")

# Tutorial Setup: Creating an instance of `CxOneClient`

If you're not familiar with the `cxone-async-api`, you should take some time to run the [first few tutorials](https://github.com/checkmarx-ts/cxone-async-api) to understand 
how to create an instance of the `CxOneClient` class.  The `CxOneClient` is used to retrieve all data needed for compiling the SARIF log.



In [ ]:
assert(server is not None and len(server) > 0)
assert(tenant_name is not None and len(server) > 0)

from cxone_api import CxOneClient,CxOneAuthEndpoint, CxOneApiEndpoint, AuthRegionEndpoints, ApiRegionEndpoints
from cxone_api.low.misc import retrieve_versions
from cxone_api.util import json_on_ok

# If the server name is set to a regional moniker, the moniker can be used to make each endpoint instance.
if server in AuthRegionEndpoints and server in ApiRegionEndpoints:
    auth_endpoint = AuthRegionEndpoints[server](tenant_name)
    api_endpoint = ApiRegionEndpoints[server]()
else:
    # The server name was provided, as would be required for single-tenant Checkmarx One, make the endpoint instances using the server name.
    # This method can also be used if you do not want to use the multi-tenant regional moniker.
    auth_endpoint = CxOneAuthEndpoint(tenant_name, server)
    api_endpoint = CxOneApiEndpoint(server)


MD(f"Auth endpoint: **{str(auth_endpoint)}**, API Endpoint: **{str(api_endpoint)}**")

if api_key is not None:
    MD("Connection will use the API key provided in the first cell.")
    # Create the client instance with the factory method for API keys
    client = CxOneClient.create_with_api_key(api_key, agent_name, auth_endpoint, api_endpoint, proxy=proxies, ssl_verify=ssl_verify)

elif oauth_client is not None and oauth_secret is not None:
    # Create the client instance with the factory method for OAuth clients
    client = CxOneClient.create_with_oauth(oauth_client, oauth_secret, agent_name, auth_endpoint, api_endpoint, proxy=proxies, ssl_verify=ssl_verify)
else:
    MD(f"No Checkmarx One credentials have been provided, can't continue.")
    assert(False)


# Make a simple authenticated API call.
JSON(json_on_ok(await retrieve_versions(client)))


# Tutorial 1: Iteratively Generating SARIF Logs

The next cell shows a code sample that creates a SARIF log for each scan id added to the `scan_ids` list in the first cell.  The log is written to a file and is optionally displayed
in the cell output as formatted JSON.  Each log generation uses the `cxone_sarif.opts.DEFAULT` report options.

This example waits until one SARIF log is fully generated before starting to generate the next log.  While this may be slow, it may be a good way to avoid API call failures
that may occur when the Checkmarx One APIs are being heavily used by other running processes.


In [ ]:
from cxone_sarif import get_sarif_v210_log_for_scan
from cxone_sarif.opts import DEFAULT

for scanid in scan_ids:
  MD(f"## Creating log for scan {scanid}")
  l = await get_sarif_v210_log_for_scan(client, DEFAULT, scanid)
  write_log(f"{scanid}.sarif", l)
  JSON(l)


# Tutorial 2: Modifying Report Options

The next cell's example works the same as the example for Tutorial 1.  The difference is that SARIF logs with only SAST results are generated.  The type of engine
results include in the generated log can be controlled by creating an instance of `cxone_sarif.opts.ReportOpts`.




In [ ]:
from cxone_sarif import get_sarif_v210_log_for_scan
from cxone_sarif.opts import ReportOpts, SastOpts

opts = ReportOpts(SkipSca=True, SkipKics=True, SkipContainers=True, SastOpts=SastOpts(SkipSast=False, OmitApiResults=True, AppendSimilarityId=False))

for scanid in scan_ids:
  MD(f"## Creating log for scan {scanid}")
  l = await get_sarif_v210_log_for_scan(client, opts, scanid)
  write_log(f"{scanid}.sarif", l)
  JSON(l)

# Tutorial 3: Generating SARIF Logs in Parallel

The next tutorial shows an advanced method of generating multiple SARIF logs using parallel execution.

Note that compiling SARIF logs makes heavy use of the Checkmarx One APIs to retrieve the data that is transformed into the SARIF log.  Expecting
a remote API endpoint to have unlimited resources to concurrently handle all API requests is not a reasonable expectation.  API failures are retried and usually
are successful with some time delays between retries.


In [ ]:
from asyncio import Semaphore, Lock
from tqdm.asyncio import tqdm
from cxone_sarif import get_sarif_v210_log_for_scan
from cxone_sarif.opts import DEFAULT

# Use a Semaphore to limit the number of concurrent SARIF logs being generated.
threads = Semaphore(2)

# Use a Lock to allow a thread exclusive access to write the log to the notebook output
display_lock = Lock()

# Create a coroutine that executes SARIF log collection concurrently
async def make_log(scanid):
  async with threads:
    l = await get_sarif_v210_log_for_scan(client, DEFAULT, scanid)
    write_log(f"{scanid}.sarif", l)
    async with display_lock:
      if display_sarif:
        MD(f"## SARIF log for scan {scanid}")
        JSON(l)    

# Use a list comprehension to allow asyncio.gather (wrapped by tqdm.asyncio.gather)
# to generate multiple SARIF logs in parallel.  The tqdm library is used only to show a progress bar.
_ = await tqdm.gather(*[make_log(scanid) for scanid in scan_ids])
